# Bronze Layer — Garmin Data Ingestion
Loads raw Garmin data from SQLite export into a Delta table in the landing volume.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS garmin_lakehouse;
CREATE SCHEMA IF NOT EXISTS garmin_lakehouse.raw;
CREATE VOLUME IF NOT EXISTS garmin_lakehouse.raw.landing;

In [0]:
%pip install garminconnect

## Bronze Layer — Garmin Activity Ingestion
Connects to Garmin API using credentials stored in Databricks Secrets,
fetches activities and writes them as a Delta table to Unity Catalog.

In [0]:
import pandas as pd
from garminconnect import Garmin
import json

In [0]:
# Retrieve credentials from Databricks Secrets
email = dbutils.secrets.get(scope="garmin", key="email")
password = dbutils.secrets.get(scope="garmin", key="password")

print("Credentials loaded ✅")

In [0]:
# Connect to Garmin API
api = Garmin(email=email, password=password)
api.login()

print("Garmin API connected ✅")

In [0]:
# Fetch activities from Garmin API
activities = api.get_activities(0, 1000)

# Convert to pandas DataFrame
df = pd.DataFrame(activities)

print(f"Fetched {len(df)} activities ✅")
print(f"Columns: {len(df.columns)}")
df.head(3)

In [0]:
print("Columns: len(df.columns)")

In [0]:
# Serialise complex fields to JSON strings
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
        df[col] = df[col].apply(
            lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
        )

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Write to Delta table in Unity Catalog
spark_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("garmin_lakehouse.raw.activities")

print(f"Written {spark_df.count()} rows to garmin_lakehouse.raw.activities ✅")

In [0]:
# Verify the table
df_check = spark.table("garmin_lakehouse.raw.activities")
print(f"Rows: {df_check.count()}")
print(f"Columns: {len(df_check.columns)}")
df_check.display()

In [0]:
# Find where the table is stored
details = spark.sql("DESCRIBE DETAIL garmin_lakehouse.raw.activities")
display(details)

In [0]:
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "garmin_lakehouse.raw.activities")
display(dt.history())